# Bronze Layer

## Data Validation

Purpose:

- Read Bronze datasets
- Validate schema
- Check null values
- Verify duplicate records
- Generate quality metrics

## Imports

In [ ]:
from common.config import StorageConfig
from common.readers import read_dataset
from common.spark import get_spark

from pyspark.sql.functions import col
from pyspark.sql.functions import count

## Spark Session

In [ ]:
spark = get_spark("Bronze Validation")

## Read Bronze Dataset

In [ ]:
customers_df = read_dataset(
    spark=spark,
    path=StorageConfig.bronze("customers"),
)

## Display Schema

In [ ]:
customers_df.printSchema()

## Display Sample

In [ ]:
customers_df.show(10, truncate=False)

## Total Records

In [ ]:
total_records = customers_df.count()

print(f"Total records: {total_records}")

## Null Validation

In [ ]:
null_counts = customers_df.select(
    [
        count(col(column).isNull() | col(column).eqNullSafe("")).alias(column)
        for column in customers_df.columns
    ]
)

null_counts.show(truncate=False)

## Duplicate Validation

In [ ]:
duplicate_df = customers_df.groupBy("customer_id").count().filter(col("count") > 1)

duplicate_df.show()

## Summary

In [ ]:
print("=" * 80)

print(f"Dataset...........: customers")
print(f"Total Records.....: {total_records}")
print(f"Duplicate Records.: {duplicate_df.count()}")

print("=" * 80)